# BIO-NN Experiment 4: Full Visualization Dashboard

Comprehensive visualization of all neuron types, spike patterns, membrane dynamics,
weight distributions, and network statistics.

### 4.1 Collect Data from All Neuron Types

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from bio_nn.neurons import create_neuron

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

NEURON_TYPES = ["lif", "adaptive_lif", "izhikevich", "dual_lif", "adex"]
all_data = {}

for name in NEURON_TYPES:
    neuron = create_neuron(name, 128).to(device)
    state = neuron._get_initial_state(8, device)
    spikes_list, mem_list = [], []

    for t in range(150):
        x = torch.randn(8, 128).to(device) * 0.8
        spikes, mem, state = neuron(x, state)
        spikes_list.append(spikes.cpu())
        mem_list.append(mem.cpu())

    all_data[name] = {
        "spikes": torch.stack(spikes_list),
        "membrane": torch.stack(mem_list)
    }
    print(f"  Collected: {name}")

print(f"Data collected for {len(all_data)} neuron types")

### 4.2 Spike Raster Comparison (All Neuron Types)

In [ ]:
n = len(all_data)
fig, axes = plt.subplots(n, 2, figsize=(14, 3 * n))

for i, (name, data) in enumerate(all_data.items()):
    # Spike raster
    spike_matrix = data["spikes"][:, 0, :64].numpy().T
    axes[i, 0].imshow(spike_matrix, aspect='auto', cmap='hot', interpolation='nearest')
    axes[i, 0].set_ylabel(name, fontweight='bold', fontsize=10)
    axes[i, 0].set_xticks([])
    if i == 0:
        axes[i, 0].set_title("Spike Raster", fontweight='bold', fontsize=12)

    # Membrane potential
    for j in range(min(4, data["membrane"].shape[2])):
        axes[i, 1].plot(data["membrane"][:, 0, j].numpy(), linewidth=0.8, label=f'n{j}')
    axes[i, 1].set_xlim([0, 150])
    axes[i, 1].legend(loc='upper right', fontsize=7, ncol=2)
    axes[i, 1].grid(True, alpha=0.2)
    if i == 0:
        axes[i, 1].set_title("Membrane Potential", fontweight='bold', fontsize=12)
    axes[i, 1].set_xlabel("Time Step" if i == n - 1 else "")

plt.suptitle("Neuron Dynamics Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('full_spike_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Spike Rate Heatmaps

In [ ]:
fig, axes = plt.subplots(1, len(all_data), figsize=(4 * len(all_data), 4))
if len(all_data) == 1:
    axes = [axes]

for i, (name, data) in enumerate(all_data.items()):
    # Spike rate per neuron over time
    rates = data["spikes"].float()  # (T, batch, neurons)
    rates_per_neuron = rates.mean(dim=1).numpy()  # (T, neurons)

    im = axes[i].imshow(rates_per_neuron.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel("Time Step")
    axes[i].set_ylabel("Neuron Index" if i == 0 else "")
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.suptitle("Spike Rate Heatmaps (per neuron)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('spike_rate_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Membrane Potential Phase Portrait

In [ ]:
fig, axes = plt.subplots(1, len(all_data), figsize=(5 * len(all_data), 5))
if len(all_data) == 1:
    axes = [axes]

for i, (name, data) in enumerate(all_data.items()):
    mem = data["membrane"][:, 0, 0].numpy()  # First neuron, first batch

    # Phase portrait: V(t) vs V(t+1)
    axes[i].plot(mem[:-1], mem[1:], 'b-', alpha=0.5, linewidth=0.5)
    axes[i].scatter(mem[0], mem[1], c='red', s=50, zorder=5, label='start')
    axes[i].scatter(mem[-1], mem[-2], c='green', s=50, zorder=5, label='end')
    axes[i].set_title(f"{name} Phase Portrait", fontweight='bold')
    axes[i].set_xlabel("V(t)")
    axes[i].set_ylabel("V(t+1)")
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Membrane Potential Phase Portraits", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('phase_portraits.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Inter-Spike Interval (ISI) Distribution

In [ ]:
fig, axes = plt.subplots(1, len(all_data), figsize=(4 * len(all_data), 4))
if len(all_data) == 1:
    axes = [axes]

for i, (name, data) in enumerate(all_data.items()):
    spikes = data["spikes"][:, 0, 0].numpy()  # First neuron
    spike_times = np.where(spikes > 0.5)[0]

    if len(spike_times) > 1:
        isi = np.diff(spike_times)
        axes[i].hist(isi, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        axes[i].set_title(f"{name}\nISI (mean={np.mean(isi):.2f})", fontweight='bold')
        axes[i].set_xlabel("ISI (steps)")
        axes[i].set_ylabel("Count")
    else:
        axes[i].text(0.5, 0.5, 'Too few spikes', ha='center', va='center',
                     transform=axes[i].transAxes)
        axes[i].set_title(name, fontweight='bold')

plt.suptitle("Inter-Spike Interval Distributions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('isi_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Network Statistics Summary

In [ ]:
print("=" * 70)
print("NETWORK STATISTICS SUMMARY")
print("=" * 70)
print(f"{'Model":<20} {'Spike Rate':<12} {'Mem Mean':<12} {'Mem Std':<12} {'ISI Mean':<12}")
print("-" * 70)

for name, data in all_data.items():
    spikes = data["spikes"]
    mem = data["membrane"]
    
    spike_rate = spikes.float().mean().item()
    mem_mean = mem.mean().item()
    mem_std = mem.std().item()
    
    # ISI for first neuron
    spike_times = np.where(spikes[:, 0, 0].numpy() > 0.5)[0]
    isi_mean = np.mean(np.diff(spike_times)) if len(spike_times) > 1 else float('nan')
    
    print(f"{name:<20} {spike_rate:<12.4f} {mem_mean:<12.4f} {mem_std:<12.4f} {isi_mean:<12.2f}")

print("=" * 70)

### 4.7 Download All Plots

In [ ]:
from google.colab import files
for f in ['full_spike_comparison.png', 'spike_rate_heatmaps.png',
          'phase_portraits.png', 'isi_distributions.png']:
    try:
        files.download(f)
    except:
        print(f"{f} not found")